# Historical notebook archive
The legacy dataset and cached outputs are excluded while provenance and redistribution rights are unresolved. Set `DIABETES_DATASET` to your own authorized CSV with the original schema to inspect the historical workflow. This notebook has not been rerun or revalidated and is not a clinical tool.
The historical scaler is fitted before the split; reproduction does not establish unbiased validation. A new evaluation should contain learned preprocessing within training folds.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
from pathlib import Path
import os

dataset_path = os.environ.get("DIABETES_DATASET")
if not dataset_path or not Path(dataset_path).is_file():
    raise FileNotFoundError("Set DIABETES_DATASET to an authorized local CSV. The legacy dataset is excluded pending provenance verification.")
df = pd.read_csv(dataset_path)
df.head()

In [ ]:
_ = df.hist(figsize=(12,10))

In [ ]:
sns.pairplot(df, hue='Diabetes')

In [ ]:
sns.heatmap(df.corr(), annot = True)

In [ ]:
from sklearn.preprocessing import StandardScaler
from keras.utils import to_categorical

In [ ]:
sc = StandardScaler()
X = sc.fit_transform(df.drop('Diabetes', axis=1))
y = df['Diabetes'].values
y_cat = to_categorical(y)

In [ ]:
X[:10]

In [ ]:
y_cat[:10]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=22)


In [ ]:
from keras.layers import Dense, Input
from keras.models import Model, Sequential
from keras.optimizers import Adam



In [ ]:
model = Sequential()
model.add(Input(shape=(8,)))
model.add(Dense(32, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(2, activation="softmax"))
model.compile(Adam(learning_rate=0.05), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
model.fit(X_train, y_train, epochs=20, verbose=2, validation_split=0.1)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
y_test_class = np.argmax(y_test, axis=1)
y_pred_class = np.argmax(y_pred, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
accuracy_score(y_test_class,y_pred_class)

In [ ]:
print(classification_report(y_test_class, y_pred_class))

In [ ]:
confusion_matrix(y_test_class, y_pred_class)